# ATiG 2026: Mendelian randomisation
8 October 2026 · About 60 minutes · Work in pairs

We will use BMI and coronary heart disease GWAS results to estimate a causal effect,
then check what happens when the MR assumptions fail.

Run the cells in order. Change the settings marked `# TRY` and discuss the questions.
The data are included. [Open in Colab](https://colab.research.google.com/github/bvilhjal/ATIG_2026/blob/main/teaching_days/2026-10-08/MR/MR_practical.ipynb)
and choose File → Save a copy in Drive. Use a Python 3 runtime.

## Setup
Run both cells. No editing needed.

In [ ]:
import io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['figure.dpi'] = 110
pd.set_option('display.precision', 4)

In [ ]:
#@title Load the prepared data (no editing needed)
CSV_DATA = """snp,effect_allele,other_allele,bmi_effect,bmi_se,chd_effect,chd_se
rs977747,G,T,-0.0168,0.003,-0.013896,0.0096146
rs17381664,C,T,0.0201,0.0031,0.016188,0.0102099
rs7550711,T,C,0.0659,0.0087,-0.048354,0.0305635
rs2820292,C,A,0.0181,0.0029,0.017948,0.0092888
rs7531118,C,T,0.0331,0.003,0.017675,0.0097331
rs543874,G,A,0.0497,0.0037,0.0076,0.0117296
rs657452,G,A,-0.0227,0.0031,-0.00767,0.0093844
rs11165643,T,C,0.0221,0.003,0.00515,0.0092844
rs1460676,C,T,0.0209,0.0038,-0.003793,0.0122576
rs1528435,T,C,0.0175,0.003,0.018036,0.0097863
rs1016287,C,T,-0.0228,0.0033,-0.014087,0.0105738
rs2890652,C,T,0.0279,0.0049,0.004026,0.0136382
rs13021737,G,A,0.0604,0.0039,0.037025,0.0124444
rs6713510,A,G,0.0164,0.0029,-0.037794,0.0095125
rs10182181,G,A,0.0309,0.0029,0.018295,0.009279
rs12986742,C,T,0.0207,0.0036,0.00822,0.0093952
rs17203016,G,A,0.0211,0.0038,0.020352,0.0122315
rs7599312,A,G,-0.0214,0.0033,-0.026653,0.011193
rs3849570,A,C,0.0183,0.0033,0.028293,0.0096396
rs6804842,G,A,0.0183,0.003,0.00994,0.0092649
rs2365389,T,C,-0.0195,0.003,-0.014975,0.0096038
rs13078960,G,T,0.029,0.0038,-0.020015,0.012616
rs16851483,T,G,0.0478,0.0075,0.014404,0.0164441
rs1516725,C,T,0.0448,0.0044,0.008179,0.0137697
rs13130484,T,C,0.0398,0.003,0.030987,0.0093463
rs13107325,T,C,0.0472,0.0066,-0.00667,0.0223469
rs11727676,C,T,-0.0365,0.0063,0.036454,0.020208
rs2112347,G,T,-0.0254,0.003,-0.005855,0.0096581
rs7715256,T,G,-0.0168,0.0029,-0.002363,0.0095842
rs6457796,C,T,0.0209,0.0033,0.033698,0.0106292
rs943005,T,C,0.0444,0.0038,0.017756,0.0117188
rs2033529,G,A,0.0183,0.0032,-0.006297,0.0104609
rs13201877,G,A,0.0236,0.0043,0.029827,0.014328
rs13191362,G,A,-0.0285,0.0047,-0.036681,0.0152568
rs9374842,T,C,0.0196,0.0034,0.013555,0.0106053
rs3800229,T,G,0.0175,0.0032,0.003892,0.009884
rs1167827,G,A,0.02,0.0031,0.011586,0.0098352
rs2060604,C,T,-0.0203,0.003,-0.019358,0.0092116
rs2183825,C,T,0.0241,0.0032,0.022607,0.0100035
rs4740619,C,T,-0.017,0.0029,-0.006546,0.0092497
rs1928295,C,T,-0.0182,0.0029,-0.006674,0.0092095
rs6477694,T,C,-0.0169,0.003,-0.006108,0.009418
rs10733682,G,A,-0.0188,0.003,-0.004541,0.0095245
rs7903146,T,C,-0.0235,0.0033,0.033092,0.0103902
rs7899106,G,A,0.0379,0.0067,-0.006811,0.0233702
rs17094222,C,T,0.0249,0.0037,0.003571,0.0116601
rs2176598,C,T,-0.0185,0.0033,-0.026064,0.0111278
rs3817334,T,C,0.0256,0.003,0.000355,0.0095386
rs12286929,G,A,0.0211,0.0029,0.001802,0.0094202
rs10840100,G,A,0.0206,0.003,0.014599,0.0095234
rs11030104,G,A,-0.0416,0.0037,-0.027672,0.0109855
rs7138803,A,G,0.032,0.003,0.008175,0.0094745
rs11057405,A,G,-0.0304,0.0053,0.013737,0.0191598
rs1441264,A,G,0.0172,0.0031,0.005096,0.0096603
rs12429545,A,G,0.0324,0.0044,0.010619,0.0129343
rs9540493,G,A,-0.0182,0.0031,-0.005837,0.0095006
rs7144011,T,G,0.0274,0.0035,0.021424,0.0120707
rs10132280,A,C,-0.0221,0.0033,-0.012169,0.010664
rs13329567,T,C,-0.0307,0.0035,-0.014147,0.0106879
rs3736485,G,A,-0.016,0.0029,0.001591,0.0095777
rs12448257,A,G,0.0246,0.0037,0.009661,0.0113483
rs879620,T,C,0.0244,0.0039,-0.00294,0.009732
rs9926784,C,T,-0.0249,0.0038,0.003405,0.011294
rs4889606,G,A,-0.0187,0.003,-0.006668,0.0102697
rs1421085,C,T,0.0803,0.003,0.02966,0.0096277
rs3888190,A,C,0.0311,0.003,0.019161,0.0101002
rs12940622,A,G,-0.0183,0.0029,-0.01627,0.0094539
rs1000940,G,A,0.0184,0.0033,-0.014538,0.0098395
rs6567160,C,T,0.0562,0.0035,0.057739,0.0105597
rs17066856,C,T,-0.0371,0.005,-0.037871,0.0154925
rs891389,T,C,0.0209,0.0037,0.025213,0.0101654
rs11672660,T,C,-0.0339,0.0038,-0.038968,0.0125585
rs17724992,G,A,-0.0196,0.0034,-0.013899,0.0102186
rs6091540,T,C,-0.0185,0.0033,-0.002356,0.0101524
rs2836754,C,T,0.0169,0.003,-0.007723,0.0095191
"""
data = pd.read_csv(io.StringIO(CSV_DATA))
print(f'Loaded {len(data)} SNPs.')

## Q1. Explore the data

These are genome-wide significant BMI GWAS variants ($P<5\times10^{-8}$),
LD-clumped to retain approximately independent signals. Each row is one SNP.

- `effect_allele`, `other_allele`: the original allele coding in the supplied GWAS data.
- `bmi_effect`, `bmi_se`: BMI effect and standard error, in SD units per effect allele.
- `chd_effect`, `chd_se`: CHD effect and standard error, in log odds per effect allele.

BMI and CHD effects refer to the same effect allele; the allele labels already match
in these data. We retain the original signs. A standard error describes uncertainty
in the estimate, not variation between people.

Look at Tables 1 and 2. What does a negative BMI effect mean? Why must both GWAS use
the same effect allele? Which MR assumption does a large association statistic support?

In [ ]:
print('Table 1. The first five SNPs and their GWAS associations.')
display(data.head())  # First five SNPs.

bmi_effect = data['bmi_effect']
bmi_se = data['bmi_se']
chd_effect = data['chd_effect']
chd_se = data['chd_se']

# Approximately chi-square (1 df) under the null of no SNP–BMI association.
association_statistic = (bmi_effect / bmi_se)**2

print(f'Smallest association statistic: {association_statistic.min():.1f}')

In [ ]:
print('Data shape (rows, columns):', data.shape)

# Summarise each column across SNPs.
summary = data[['bmi_effect', 'bmi_se', 'chd_effect', 'chd_se']].agg(['min', 'median', 'max'])
print('Table 2. Smallest, middle and largest values across SNPs.')

display(summary)

# sum() counts how many SNPs satisfy the condition.
print('SNPs with negative BMI effects:', (bmi_effect < 0).sum())
print('SNPs with positive CHD effects:', (chd_effect > 0).sum())
print('SNPs with negative CHD effects:', (chd_effect < 0).sum())

## Q2. Look at one SNP

Figure 1 plots the BMI and CHD effects, with approximate 95% intervals for CHD.
The orange point is `rs7550711`, the third row of Table 1.

Find its effect estimates and standard errors in the table. Does its CHD interval
include zero? What does this tell you about its negative estimate?

In [ ]:
example_snp = data.iloc[2]  # Third row: Python counts from zero.

example_lower = example_snp['chd_effect'] - 1.96 * example_snp['chd_se']
example_upper = example_snp['chd_effect'] + 1.96 * example_snp['chd_se']

print('Highlighted SNP:', example_snp['snp'])
print(f"BMI effect: {example_snp['bmi_effect']:.4f} SD per allele")
print(f"CHD effect: {example_snp['chd_effect']:.4f} log odds per allele")
print(f'CHD 95% interval: {example_lower:.4f} to {example_upper:.4f} log odds per allele')

In [ ]:
plt.figure(figsize=(6, 4))

# Vertical bars show approximate 95% intervals for CHD effects.
plt.errorbar(bmi_effect, chd_effect, yerr=1.96 * chd_se,
             fmt='o', markersize=4, alpha=0.7)

# Highlight the SNP selected above.
plt.scatter(example_snp['bmi_effect'], example_snp['chd_effect'],
            color='darkorange', label=example_snp['snp'], zorder=3)
plt.legend()
plt.axhline(0, color='grey', linestyle='--')
plt.axvline(0, color='grey', linestyle=':')
plt.xlabel('SNP effect on BMI (SD per allele)')
plt.ylabel('SNP effect on CHD (log odds per allele)')
plt.title('Figure 1. BMI and CHD associations')
plt.tight_layout()
plt.show()

## Q3. Estimate the effect using one SNP

The Wald ratio expresses the CHD effect per SD higher BMI:

$$\text{MR estimate} = \frac{\text{SNP effect on CHD}}{\text{SNP effect on BMI}}. \qquad (1)$$

`np.exp(...)` converts log odds to an odds ratio.
Compare rows 0 and 2. Why do their estimates differ?

In [ ]:
snp_number = 0  # TRY: change to 2 (the orange point).
snp = data.iloc[snp_number]

single_snp_estimate = snp['chd_effect'] / snp['bmi_effect']
single_snp_odds_ratio = np.exp(single_snp_estimate)

print('SNP:', snp['snp'])

print(f'MR estimate: {single_snp_estimate:.3f} log odds per SD BMI')
print(f'Odds ratio: {single_snp_odds_ratio:.3f} per SD BMI')

## Q4. Combine the SNPs

Inverse-variance weighting (IVW) gives more weight to precise ratio estimates:

$$\text{IVW estimate} =
\frac{\sum_j \text{weight}_j\,\text{ratio}_j}{\sum_j \text{weight}_j}.
\qquad (2)$$

We treat the SNPs as independent and uncertainty in the BMI effects as small.
Halving a ratio's standard error gives it four times the weight. Why?
Interpret the combined odds ratio.

In [ ]:
# Ratio estimates
snp_estimates = chd_effect / bmi_effect

# Approximate standard errors; abs keeps them positive.
snp_standard_errors = chd_se / np.abs(bmi_effect)

# Inverse-variance weights
weights = 1 / snp_standard_errors**2

# Weighted mean
mr_estimate = np.average(snp_estimates, weights=weights)

print(f'IVW estimate: {mr_estimate:.3f} log odds per SD BMI')
print(f'Odds ratio: {np.exp(mr_estimate):.3f} per SD BMI')

## Q5. Calculate a confidence interval

`Q` measures disagreement between SNP estimates. With 75 SNPs and one fitted effect,
its expected value under the common-effect model is about 74. Excess disagreement
widens the interval; it does not remove bias from invalid instruments.

Does the approximate 95% interval include an odds ratio of one?
What assumptions are needed for a causal interpretation?

In [ ]:
# Weighted disagreement between SNP estimates.
residuals = snp_estimates - mr_estimate

Q = np.sum(weights * residuals**2)
expected_Q = len(data) - 1

variance_inflation = max(1, Q / expected_Q)  # Never shrink the interval.

# Allow for extra disagreement, then calculate the 95% interval.
standard_error = 1 / np.sqrt(weights.sum())

mr_standard_error = standard_error * np.sqrt(variance_inflation)

lower_limit = mr_estimate - 1.96 * mr_standard_error
upper_limit = mr_estimate + 1.96 * mr_standard_error

print(f'Variance inflation from SNP disagreement: {variance_inflation:.2f}')
print(f'Odds ratio: {np.exp(mr_estimate):.3f}')

print(f'95% interval: {np.exp(lower_limit):.3f} to {np.exp(upper_limit):.3f}')

## Q6. Add pleiotropy

Now simulate two continuous traits, with known SNP effects on the exposure.
The outcome depends on a causal pathway and a second pathway proportional to the
SNP's exposure effect. The second pathway represents horizontal pleiotropy.
The MR estimate is in outcome units per exposure unit.

Try Table 3's settings, rerunning both cells each time.
Figure 2 shows the associations and IVW fit (orange line).

Table 3. Simulation settings.

| Setting | `causal_effect` | `pleiotropy` |
|---|---:|---:|
| A: causal effect only | 0.3 | 0.0 |
| B: causal effect plus pleiotropy | 0.3 | 0.2 |
| C: pleiotropy only | 0.0 | 0.3 |

Why do A and C give identical results? Which MR assumption fails in C?

In [ ]:
causal_effect = 0.3  # TRY: the settings in Table 3.
pleiotropy = 0.0    # TRY: the settings in Table 3.

# Keep the random noise the same when changing the settings.
rng = np.random.default_rng(20261008)
exposure_effect = np.linspace(0.04, 0.12, 30)  # Known effects for 30 SNPs.
outcome_se = np.full(30, 0.012)

noise = rng.normal(0, outcome_se)

causal_path = causal_effect * exposure_effect  # Through the exposure.
other_path = pleiotropy * exposure_effect     # Through another pathway.
outcome_effect = causal_path + other_path + noise

In [ ]:
# Same IVW calculation as Q4.
simulated_ratios = outcome_effect / exposure_effect
simulated_standard_errors = outcome_se / np.abs(exposure_effect)
simulated_weights = 1 / simulated_standard_errors**2
simulated_estimate = np.average(simulated_ratios, weights=simulated_weights)
print(f'True causal effect: {causal_effect:.3f}; MR estimate: {simulated_estimate:.3f}')

plt.figure(figsize=(6, 4))
plt.scatter(exposure_effect, outcome_effect)

plt.plot(exposure_effect, simulated_estimate * exposure_effect, color='orange')
plt.xlabel('Known SNP effect on the exposure')
plt.ylabel('Estimated SNP effect on the outcome')
plt.title('Figure 2. A simulated MR analysis')
plt.tight_layout()
plt.show()

## Q7. Conclude

Write three sentences: report the BMI–CHD estimate and interval with units, state the main
assumption you cannot verify, and suggest another study that could test the conclusion.

## Data and method notes

The data are a frozen [TwoSampleMR example](https://github.com/MRCIEU/TwoSampleMR/blob/f4920457ee65c279f17c9a891173239d6adcd52c/vignettes/harmonise.Rmd):
[Locke et al. (2015), BMI](https://pubmed.ncbi.nlm.nih.gov/25673413/), and
[Nikpay et al. (2015), CHD](https://pubmed.ncbi.nlm.nih.gov/26343387/).
BMI was studied in up to 339,224 participants; CHD in 60,801 cases and 123,504 controls.
Both source datasets are labelled mixed ancestry.

The source workflow selects significant BMI associations and applies
[LD clumping](https://github.com/MRCIEU/TwoSampleMR/blob/f4920457ee65c279f17c9a891173239d6adcd52c/R/instruments.R).
Of 79 matched SNPs, we exclude four palindromic A/T or C/G variants and retain the
original GWAS effect alleles and signed effects for the remaining 75. BMI and CHD
allele labels match. Raw data and preparation details are in `data/`.

We use first-order IVW and a multiplicative random-effects interval, treating SNPs
as independent and exposure effects as precisely estimated. Proxy substitutions and
participant overlap have not been independently audited. A genetic contrast is not
automatically equivalent to a treatment.

Background: [Burgess et al. (2013), summary-data MR](https://pubmed.ncbi.nlm.nih.gov/24114802/);
[Hemani et al. (2018), MR-Base](https://doi.org/10.7554/eLife.34408);
[Morrison et al. (2020), correlated pleiotropy](https://www.nature.com/articles/s41588-020-0631-4).